In [2]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [4]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [20]:
# Import datasets
datasets = {}
dataset_names = ['clfever', 'phemeplus', 'vitc']
for dataset_name in dataset_names:
    with open(f'{dataset_name}.json') as f:
        datasets[dataset_name] = json.load(f)

In [23]:
# Populate Vercel KV with datasets
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for datapoint in dataset:
        id = datapoint['claim_id']
        r.hset(id, mapping={
            'claim': datapoint['claim'],
            'evidence': datapoint['evidence'],
            'label': datapoint['label']
        })

In [45]:
# Create batches containing 25 datapoints each
batch_ids = []
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for i in range(0, len(dataset), 25):
        batch_dataset = dataset[i:i+25]
        claim_ids = []
        for batch_datapoint in batch_dataset:
            id = batch_datapoint['claim_id']
            claim_ids.append(id)
        batch_id = f'batch_{dataset_name}_{i//25 + 1}'
        batch_ids.append(batch_id)
        r.hset(batch_id, mapping={'claim_ids': json.dumps(claim_ids)})    

In [47]:
# Create queue 
r.lpush('queue', *batch_ids)

7

In [5]:
r.lrange('queue', 0, -1)

[b'batch_vitc_2',
 b'batch_vitc_1',
 b'batch_phemeplus_4',
 b'batch_phemeplus_3',
 b'batch_phemeplus_2',
 b'batch_phemeplus_1',
 b'batch_clfever_1']

In [55]:
r.lindex('queue', 6)

b'batch_clfever_1'

In [6]:
curos, keys = r.scan(cursor=0)
decoded_keys = [key.decode('utf-8') for key in keys]
decoded_keys

['batch_clfever_1',
 'batch_phemeplus_1',
 'batch_phemeplus_2',
 'batch_phemeplus_3',
 'batch_phemeplus_4',
 'batch_vitc_1',
 'batch_vitc_2',
 'cfever_0',
 'cfever_1',
 'cfever_10']

In [7]:
r.hgetall('vitc_21')

{b'label': b'REFUTES',
 b'claim': b'Kesari ( film ) has a domestic gross of under 161 crore .',
 b'evidence': b'its domestic gross is 161.95 crore and overseas gross 22.52 crore .'}